🛠️ Task 1 — Data Loading, Merging & Deep Exploration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
os.makedirs('charts', exist_ok=True)

# 1. Load Primary Superstore Sales Dataset
df_store = pd.read_csv('train.csv')
df_store['Order Date'] = pd.to_datetime(df_store['Order Date'], format='%d/%m/%Y', errors='coerce')
df_store = df_store.dropna(subset=['Order Date', 'Sales'])

# Feature Engineering for Primary Dataset
df_store['Year'] = df_store['Order Date'].dt.year
df_store['Month'] = df_store['Order Date'].dt.month

# Aggregate primary sales to a monthly baseline
df_store_monthly = df_store.groupby(['Year', 'Month'])['Sales'].sum().reset_index()
# Create a proper first-of-the-month datetime index for tracking
df_store_monthly['Date'] = pd.to_datetime(df_store_monthly[['Year', 'Month']].assign(Day=1))


# 2. Load Supplementary Video Game Sales Dataset
df_games = pd.read_csv('vgsales.csv')

# Clean Year column since it contains some N/A strings, then cast to integer
df_games = df_games.dropna(subset=['Year'])
df_games['Year'] = df_games['Year'].astype(int)

# Group game sales by Year to look at global entertainment demand trends
df_games_annual = df_games.groupby('Year')['Global_Sales'].sum().reset_index()
df_games_annual.rename(columns={'Global_Sales': 'External_Gaming_Market_Demand'}, inplace=True)


# 3. Multi-Source Analysis: Merging Datasets
# Merge the historical annual industry demand trends onto your granular store trends
df_merged = pd.merge(df_store_monthly, df_games_annual, on='Year', how='left')

print("=== TASK 1: MULTI-SOURCE DATA INTELLIGENCE COMPILED ===")
print(f"Merged Dataset Shape: {df_merged.shape[0]} months across mapped years.\n")
display(df_merged.head())

# 4. Core Descriptive Questions
print("--- Analytical Explorations ---")
revenue_by_cat = df_store.groupby('Category')['Sales'].sum().sort_values(ascending=False)
print("1. Highest Total Revenue Generating Category (Superstore):")
print(revenue_by_cat.apply(lambda x: f"{x:,.2f}"))